# Join and Transformation

Combine the validated university datasets and apply the agreed transformation rules.

In [1]:
from pathlib import Path
import pandas as pd

project_root = Path("..")
interim_dir = project_root / "data" / "interim"
processed_dir = project_root / "data" / "processed"

In [2]:
kaust = pd.read_csv(
    interim_dir / "KAUST_validated.csv"
)

kfupm = pd.read_csv(
    interim_dir / "KFUPM_validated.csv"
)

print("KAUST:", kaust.shape)
print("KFUPM:", kfupm.shape)

KAUST: (1042, 14)
KFUPM: (48, 14)


In [3]:
combined = pd.concat(
    [kaust, kfupm],
    ignore_index=True
)

print("Rows before combine:", len(kaust) + len(kfupm))
print("Rows after combine:", len(combined))
print("Duplicate research IDs:", combined["research_id"].duplicated().sum())

Rows before combine: 1090
Rows after combine: 1090
Duplicate research IDs: 0


In [4]:
combined = combined.drop(
    columns="validation_error",
    errors="ignore"
)

print("Final columns:", len(combined.columns))

Final columns: 13


In [5]:
transformation_rules = pd.DataFrame([
    [
        "R1",
        "Convert publication_year to nullable integer",
        "publication_year",
        "publication_year"
    ],
    [
        "R2",
        "Create a flag showing whether a DOI is available",
        "doi",
        "has_doi"
    ],
    [
        "R3",
        "Calculate the number of words in each abstract",
        "abstract",
        "abstract_word_count"
    ]
], columns=[
    "rule_id",
    "description",
    "input_columns",
    "output_column"
])

transformation_rules

,rule_id,description,input_columns,output_column
0,R1,Convert publication_year to nullable integer,publication_year,publication_year
1,R2,Create a flag showing whether a DOI is available,doi,has_doi
2,R3,Calculate the number of words in each abstract,abstract,abstract_word_count


In [6]:
final_data = combined.copy()

final_data["publication_year"] = pd.to_numeric(
    final_data["publication_year"],
    errors="coerce"
).astype("Int64")

final_data["has_doi"] = (
    final_data["doi"]
    .notna()
    & final_data["doi"].astype("string").str.strip().ne("")
)

final_data["abstract_word_count"] = (
    final_data["abstract"]
    .fillna("")
    .astype("string")
    .str.split()
    .str.len()
)

print("Final rows:", len(final_data))
print("Final columns:", len(final_data.columns))

Final rows: 1090
Final columns: 15


In [7]:
assert len(final_data) == len(combined)

assert final_data["research_id"].duplicated().sum() == 0

assert final_data["publication_year"].notna().all()

assert final_data["has_doi"].isin([True, False]).all()

assert (final_data["abstract_word_count"] >= 0).all()

print("All transformation tests passed.")

All transformation tests passed.


In [9]:
processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_file = processed_dir / "final_rana.csv"

final_data.to_csv(
    output_file,
    index=False
)

saved_final = pd.read_csv(output_file)

print("Saved:", output_file)
print("Saved rows:", len(saved_final))
print("Saved columns:", len(saved_final.columns))

Saved: ..\data\processed\final_rana.csv
Saved rows: 1090
Saved columns: 15
